# AlphaZero vs KataGo — 19×19 Game Replay

Plays one full game of the trained 19×19 AlphaZero engine against a pretrained KataGo model,
then plots every board position.

In [ ]:
import os
import subprocess
import numpy as np
import math
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from dataclasses import dataclass
from typing import List, Optional

BOARD_SIZE    = 19
NUM_POSITIONS = BOARD_SIZE * BOARD_SIZE   # 361
PASS_ACTION   = NUM_POSITIONS             # 361
ACTION_SIZE   = NUM_POSITIONS + 1         # 362
KOMI          = 7.5
# GTP column letters: A-H then J (skip I), then K-T for 19x19
GTP_COLS      = 'ABCDEFGHJKLMNOPQRST'
MAX_MOVES     = BOARD_SIZE * BOARD_SIZE * 3

_HERE       = os.path.dirname(os.path.abspath('__file__'))  # go/notebooks/
_GO_DIR     = os.path.join(_HERE, '..')                     # go/
FIGURES_DIR = os.path.join(_HERE, 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)

KATAGO_BIN  = os.path.join(_GO_DIR, 'KataGo', 'cpp', 'katago')
GTP_CONFIG  = os.path.join(_GO_DIR, 'KataGo', 'cpp', 'configs', 'gtp_example.cfg')
GTP_ENGINE  = os.path.join(_GO_DIR, 'gtp_engine_19')
MODELS_DIR  = os.path.join(_GO_DIR, 'models_19x19_base-pass')
KATAGO_PRETRAINED_DIR = os.path.join(_GO_DIR, 'pretrained_katago_models')

print(f'go/            → {os.path.abspath(_GO_DIR)}')
print(f'figures/       → {FIGURES_DIR}')
print(f'gtp_engine_19  : {os.path.exists(GTP_ENGINE)}')
print(f'katago         : {os.path.exists(KATAGO_BIN)}')

## 1 — Configuration

Edit these values before running the rest of the notebook.

In [ ]:
# ── AlphaZero ───────────────────────────────────────────────────────────────────────────
# Leave AZ_MODEL_PATH = None to auto-pick the latest _ts.pt in models_19x19_base/
AZ_MODEL_PATH  = None
AZ_SIMS        = 160     # MCTS simulations per move
AZ_ADD_NOISE   = True     # Add AlphaZero root Dirichlet noise during notebook play
AZ_NOISE_ALPHA = 0.03 * BOARD_SIZE  # 0.57 on 19x19
AZ_NOISE_FRAC  = 0.35     # Default training value is 0.25; 0.35 adds a little more noise

# ── KataGo ──────────────────────────────────────────────────────────────────────────
# Available ELOs: 482, 802, 10000, 11075, 11177, 11367, 11554, 13271
KATAGO_ELO    = 482
KATAGO_VISITS = 10     # KataGo search visits per move

# ── Match ────────────────────────────────────────────────────────────────────────────
AZ_COLOR = -1   # 1 = AlphaZero plays Black, -1 = AlphaZero plays White

# ── Resolve paths ───────────────────────────────────────────────────────────────────
if AZ_MODEL_PATH is None:
    _ts_files = sorted(
        [f for f in os.listdir(MODELS_DIR) if f.endswith('_ts.pt')
         and not f.startswith('-')],
        key=lambda f: int(f.split('_')[0])
    )
    if not _ts_files:
        raise FileNotFoundError(f'No *_ts.pt models found in {MODELS_DIR}')
    AZ_MODEL_PATH = os.path.join(MODELS_DIR, _ts_files[-1])

KATAGO_MODEL_PATH = os.path.join(KATAGO_PRETRAINED_DIR, f'katago-elo-{KATAGO_ELO}.gz')

if not os.path.exists(AZ_MODEL_PATH):
    raise FileNotFoundError(f'AlphaZero model not found: {AZ_MODEL_PATH}')
if not os.path.exists(KATAGO_MODEL_PATH):
    raise FileNotFoundError(f'KataGo model not found: {KATAGO_MODEL_PATH}')

az_label = 'Black' if AZ_COLOR == 1 else 'White'
kg_label = 'White' if AZ_COLOR == 1 else 'Black'
print(f'AlphaZero ({az_label}): {os.path.basename(AZ_MODEL_PATH)}')
print(f'KataGo    ({kg_label}): katago-elo-{KATAGO_ELO}')

## 2 — Go engine (board tracking)

Mirrors `go_engine.c` — used to maintain board state for visualisation.
The two GTP engines handle their own internal state; this is purely for display.

In [ ]:
NEIGHBORS: List[List[int]] = []
for _i in range(NUM_POSITIONS):
    _r, _c = _i // BOARD_SIZE, _i % BOARD_SIZE
    _nb = []
    if _r > 0:             _nb.append((_r-1)*BOARD_SIZE + _c)
    if _r < BOARD_SIZE-1:  _nb.append((_r+1)*BOARD_SIZE + _c)
    if _c > 0:             _nb.append(_r*BOARD_SIZE + (_c-1))
    if _c < BOARD_SIZE-1:  _nb.append(_r*BOARD_SIZE + (_c+1))
    NEIGHBORS.append(_nb)


def find_group(board, idx):
    color = board[idx]
    group, in_stack, liberty_seen = [], [False]*NUM_POSITIONS, [False]*NUM_POSITIONS
    stack = [idx]; in_stack[idx] = True; libs = 0
    while stack:
        cur = stack.pop(); group.append(cur)
        for nb in NEIGHBORS[cur]:
            if board[nb] == 0:
                if not liberty_seen[nb]: liberty_seen[nb] = True; libs += 1
            elif board[nb] == color and not in_stack[nb]:
                in_stack[nb] = True; stack.append(nb)
    return group, libs


def capture_dead_stones(board, player):
    opp = -player; captured = 0; visited = [False]*NUM_POSITIONS
    for idx in range(NUM_POSITIONS):
        if board[idx] != opp or visited[idx]: continue
        group, libs = find_group(board, idx)
        for pos in group: visited[pos] = True
        if libs == 0:
            for pos in group: board[pos] = 0
            captured += len(group)
    return captured


def is_suicide(board, idx, player):
    test = board.copy(); test[idx] = player
    for nb in NEIGHBORS[idx]:
        if test[nb] == -player:
            _, libs = find_group(test, nb)
            if libs == 0: return False
    _, libs = find_group(test, idx)
    return libs == 0


@dataclass
class GoState:
    board:              np.ndarray
    ko_point:           int
    consecutive_passes: int

    @staticmethod
    def initial():
        return GoState(np.zeros(NUM_POSITIONS, dtype=np.int8), -1, 0)

    def copy(self):
        return GoState(self.board.copy(), self.ko_point, self.consecutive_passes)


def go_next_state_canonical(state: GoState, action: int) -> GoState:
    ns = state.copy()
    if action == PASS_ACTION:
        ns.ko_point = -1; ns.consecutive_passes += 1; ns.board = -ns.board; return ns
    ns.consecutive_passes = 0; ns.board[action] = 1
    captured = capture_dead_stones(ns.board, 1)
    ko_point = -1
    if captured == 1:
        group, libs = find_group(ns.board, action)
        if len(group) == 1 and libs == 1:
            for nb in NEIGHBORS[action]:
                if ns.board[nb] == 0: ko_point = nb; break
    ns.ko_point = ko_point; ns.board = -ns.board; return ns


def go_game_ended(state: GoState) -> bool:
    return state.consecutive_passes >= 2


def count_territory(board_abs):
    bs = int(np.sum(board_abs == 1)); ws = int(np.sum(board_abs == -1))
    visited = (board_abs != 0).tolist()
    for start in range(NUM_POSITIONS):
        if visited[start]: continue
        region, bb, wb, stack = [], False, False, [start]
        visited[start] = True
        while stack:
            cur = stack.pop(); region.append(cur)
            for nb in NEIGHBORS[cur]:
                if   board_abs[nb] ==  1: bb = True
                elif board_abs[nb] == -1: wb = True
                elif not visited[nb]:     visited[nb] = True; stack.append(nb)
        if bb and not wb:  bs += len(region)
        elif wb and not bb: ws += len(region)
    return bs, ws


def go_get_winner(state: GoState, player: int) -> int:
    board_abs = state.board.copy()
    if player == -1: board_abs = -board_abs
    bs, ws = count_territory(board_abs)
    wt = ws + KOMI
    return 1 if bs > wt else (-1 if wt > bs else 0)


def score_string(state: GoState, player: int) -> str:
    board_abs = state.board.copy()
    if player == -1: board_abs = -board_abs
    bs, ws = count_territory(board_abs)
    wt = ws + KOMI
    diff = bs - wt
    if   diff > 0: return f'B+{diff:.1f}'
    elif diff < 0: return f'W+{-diff:.1f}'
    else:          return 'Draw'


print('Go engine ready')

## 3 — Coordinate helpers

In [ ]:
def az_to_gtp(action: int) -> str:
    """AlphaZero action index → GTP vertex string (e.g. 0 → 'A19', 361 → 'pass')."""
    if action == PASS_ACTION:
        return 'pass'
    row = action // BOARD_SIZE
    col = action % BOARD_SIZE
    return f'{GTP_COLS[col]}{BOARD_SIZE - row}'


def gtp_to_az(gtp_move: str) -> Optional[int]:
    """GTP vertex → AlphaZero action index. Returns None on resign."""
    gtp_move = gtp_move.strip().upper()
    if gtp_move in ('PASS', ''): return PASS_ACTION
    if gtp_move == 'RESIGN':     return None
    col = GTP_COLS.index(gtp_move[0])
    row = BOARD_SIZE - int(gtp_move[1:])
    return row * BOARD_SIZE + col


# Sanity checks
assert az_to_gtp(0)   == 'A19', f'got {az_to_gtp(0)}'    # top-left corner
assert az_to_gtp(18)  == 'T19', f'got {az_to_gtp(18)}'   # top-right corner
assert az_to_gtp(342) == 'A1',  f'got {az_to_gtp(342)}'  # bottom-left corner
assert az_to_gtp(360) == 'T1',  f'got {az_to_gtp(360)}'  # bottom-right corner
assert az_to_gtp(361) == 'pass'
assert gtp_to_az('A19') == 0
assert gtp_to_az('K10') == 180   # centre of 19x19 (row 9, col 9)
assert gtp_to_az('pass') == PASS_ACTION
assert gtp_to_az('resign') is None
print('Coordinate helpers OK')

## 4 — GTP engine wrapper

A single class that works for both `gtp_engine_19` (AlphaZero) and `katago gtp` (KataGo).
Both speak the same GTP v2 protocol.

In [ ]:
class GTPEngine:
    """Subprocess wrapper for any GTP v2 engine."""

    def __init__(self, cmd: list, board_size: int = BOARD_SIZE, komi: float = KOMI,
                 label: str = ''):
        self.label = label
        self.proc  = subprocess.Popen(
            cmd,
            stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.PIPE,
            text=True, bufsize=1,
        )
        self._cmd(f'boardsize {board_size}')
        self._cmd(f'komi {komi}')
        self._cmd('clear_board')

    def _cmd(self, command: str) -> str:
        """Send a GTP command; return the response text (stripped)."""
        self.proc.stdin.write(command + '\n')
        self.proc.stdin.flush()
        lines = []
        while True:
            line = self.proc.stdout.readline()
            if line == '':
                stderr_out = self.proc.stderr.read()
                msg = f'{self.label}: process exited during {repr(command)}'
                if stderr_out.strip():
                    msg += f'\n{stderr_out.strip()}'
                raise RuntimeError(msg)
            line = line.rstrip('\n')
            if line.startswith('=') or line.startswith('?'):
                lines.append(line)
                while True:
                    nxt = self.proc.stdout.readline().rstrip('\n')
                    if nxt == '': break
                    lines.append(nxt)
                break
        resp = '\n'.join(lines)
        if resp.startswith('?'):
            raise RuntimeError(f'{self.label} GTP error for {repr(command)}: {resp}')
        return resp.lstrip('=').strip()

    def play(self, color: str, gtp_vertex: str):
        """Inform the engine of a move already played."""
        self._cmd(f'play {color} {gtp_vertex}')

    def genmove(self, color: str) -> str:
        """Ask engine to pick and play a move. Returns GTP vertex (or 'resign')."""
        return self._cmd(f'genmove {color}').upper()

    def close(self):
        try: self._cmd('quit')
        except Exception: pass
        try: self.proc.wait(timeout=5)
        except Exception: self.proc.kill()


def make_az_engine() -> GTPEngine:
    import torch
    use_cuda = torch.cuda.is_available()
    cmd = [GTP_ENGINE, AZ_MODEL_PATH, '--sims', str(AZ_SIMS), '--batch', '32']
    if AZ_ADD_NOISE:
        cmd += ['--noise-alpha', str(AZ_NOISE_ALPHA), '--noise-frac', str(AZ_NOISE_FRAC)]
    if use_cuda: cmd.append('--cuda')
    return GTPEngine(cmd, label='AlphaZero')


def make_katago_engine() -> GTPEngine:
    override = f'maxVisits={KATAGO_VISITS},logToStderr=false,logDir=,allowResignation=true'
    cmd = [
        KATAGO_BIN, 'gtp',
        '-model',  KATAGO_MODEL_PATH,
        '-config', GTP_CONFIG,
        '-override-config', override,
    ]
    return GTPEngine(cmd, label='KataGo')


print('GTP wrapper ready')

## 5 — Play the game

The arbiter starts both engines, alternates `genmove` / `play` calls, and records every
board position for replay.  Both engines are shut down cleanly at the end.

In [ ]:
import time

def play_game(az_color: int = 1):
    """
    Play one full game.  az_color: +1 = AlphaZero is Black, -1 = AlphaZero is White.
    Returns list of frame dicts and final state.
    """
    az  = make_az_engine()
    kg  = make_katago_engine()

    state  = GoState.initial()
    player = 1          # absolute: 1=Black, -1=White
    frames = []         # one entry per move
    resigned = False

    print(f'Starting: AlphaZero ({"Black" if az_color==1 else "White"}) '
          f'vs KataGo-{KATAGO_ELO} ({"White" if az_color==1 else "Black"})')
    print('Playing ', end='', flush=True)

    try:
        for _ in range(MAX_MOVES):
            if go_game_ended(state):
                break

            color_str = 'b' if player == 1 else 'w'
            is_az_turn = (player == az_color)

            t0 = time.time()
            if is_az_turn:
                gtp_vertex = az.genmove(color_str)
                if gtp_vertex != 'RESIGN':
                    kg.play(color_str, gtp_vertex)
            else:
                gtp_vertex = kg.genmove(color_str)
                if gtp_vertex != 'RESIGN':
                    az.play(color_str, gtp_vertex)
            elapsed = time.time() - t0

            action = gtp_to_az(gtp_vertex)
            engine_label = 'AZ' if is_az_turn else 'KG'

            if action is None:  # resign
                print(f'\n  {engine_label} resigned at move {len(frames)+1}')
                resigned = True
                break

            frames.append({
                'move_num':     len(frames) + 1,
                'player':       player,
                'is_az':        is_az_turn,
                'gtp_vertex':   gtp_vertex,
                'action':       action,
                'state_before': state.copy(),
                'elapsed_s':    elapsed,
            })

            state  = go_next_state_canonical(state, action)
            player = -player
            print('.', end='', flush=True)

    finally:
        az.close()
        kg.close()

    print()
    return frames, state, player, resigned


# Run the game
frames, final_state, final_player, resigned = play_game(AZ_COLOR)

if resigned:
    winner_abs = -final_player   # whoever's turn it was when opponent resigned
    score_str  = f'{"B" if winner_abs==1 else "W"}+R'
else:
    winner_abs = go_get_winner(final_state, final_player)
    score_str  = score_string(final_state, final_player)

winner_name = 'Black' if winner_abs == 1 else ('White' if winner_abs == -1 else 'Draw')
az_won = (winner_abs == AZ_COLOR)

print(f'\nResult  : {score_str}  ({winner_name} wins)')
print(f'Moves   : {len(frames)}')
print(f'AlphaZero ({az_label}): {"WIN" if az_won else "LOSS"}')

## 6 — Visualisation helpers

In [ ]:
# Star points (hoshi) for 19x19: rows/cols 3, 9, 15 (0-indexed)
HOSHI = [3, 9, 15]

def draw_board(ax, state: GoState, player: int,
               title: str = '',
               last_move: int = -1,
               last_is_az: bool = True):
    """
    Draw a 19×19 Go board.
    last_move: action index of the most recently played stone.
    last_is_az: True if the last move was AlphaZero's.
      AZ moves  → cyan square on the stone
      KG moves  → cyan dot on the stone
    """
    n = BOARD_SIZE
    ax.set_facecolor('#DCB06A')
    ax.set_xlim(-0.5, n-0.5); ax.set_ylim(-0.5, n-0.5); ax.set_aspect('equal')
    ax.set_xticks(range(n)); ax.set_yticks(range(n))
    ax.set_xticklabels(list(GTP_COLS), fontsize=5)
    ax.set_yticklabels(range(1, n+1), fontsize=5)
    ax.tick_params(length=2, pad=1)
    ax.grid(color='#5C3D1E', linewidth=0.4)
    ax.set_title(title, fontsize=7, pad=2)
    for hr in HOSHI:
        for hc in HOSHI:
            ax.plot(hc, hr, 'o', color='#5C3D1E', markersize=2, zorder=2)

    abs_board = state.board * player
    for idx in range(NUM_POSITIONS):
        r, c = idx // n, idx % n
        v = abs_board[idx]
        if v == 0: continue
        sc = '#1A1A1A' if v == 1 else '#F5F5F0'
        ec = '#888888' if v == 1 else '#333333'
        ax.add_patch(plt.Circle((c, r), 0.44, color=sc, ec=ec, lw=0.5, zorder=3))
        if idx == last_move:
            if last_is_az:
                # AlphaZero: cyan square marker
                ax.add_patch(plt.Rectangle((c-0.16, r-0.16), 0.32, 0.32,
                                           color='cyan', zorder=5))
            else:
                # KataGo: dot marker
                dot_c = '#F5F5F0' if v == 1 else '#1A1A1A'
                ax.plot(c, r, 'o', color=dot_c, markersize=4, zorder=5)


print('Visualisation ready')

## 7 — Full game replay

Every board position, one subplot per move.  
**■ cyan square** = AlphaZero's last stone · **● dot** = KataGo's last stone

> **Note:** A 19×19 game may have 150–300 moves, producing a tall figure.
> Scroll down or open `figures/game_az_vs_kg_19x19.png` for the full replay.

In [ ]:
COLS = 10
SUB_W, SUB_H = 2.2, 2.5   # inches per subplot

# +2: empty board at start, final board at end
n_frames = len(frames) + 2
rows = math.ceil(n_frames / COLS)

fig, axes = plt.subplots(rows, COLS,
                         figsize=(COLS * SUB_W, rows * SUB_H))
axes = axes.flatten()

# Frame 0: empty board
draw_board(axes[0], GoState.initial(), player=1, title='Start')

for i, f in enumerate(frames):
    ax = axes[i + 1]
    engine = 'AZ' if f['is_az'] else 'KG'
    color  = 'B' if f['player'] == 1 else 'W'
    title  = f"M{f['move_num']} {engine}({color}): {f['gtp_vertex']}"

    state_after = go_next_state_canonical(f['state_before'], f['action'])
    draw_board(ax, state_after, player=-f['player'],
               title=title,
               last_move=f['action'] if f['action'] != PASS_ACTION else -1,
               last_is_az=f['is_az'])

# Final frame: scored board
draw_board(axes[len(frames) + 1], final_state, player=final_player,
           title=f'Final: {score_str}')

for ax in axes[n_frames:]: ax.axis('off')

# Legend patches
az_patch = mpatches.Patch(facecolor='cyan',  label=f'AlphaZero ({az_label})')
kg_patch = mpatches.Patch(facecolor='gray',  label=f'KataGo-{KATAGO_ELO} ({kg_label})')
fig.legend(handles=[az_patch, kg_patch], loc='lower center',
           ncol=2, fontsize=10, frameon=True,
           bbox_to_anchor=(0.5, -0.005))

result_title = (f'AlphaZero vs KataGo-{KATAGO_ELO}  —  19×19  —  '
                f'{score_str}  ({len(frames)} moves)')
plt.suptitle(result_title, fontsize=13, y=1.005)
plt.tight_layout()
out_path = os.path.join(FIGURES_DIR, 'game_az_vs_kg_19x19.png')
plt.savefig(out_path, dpi=110, bbox_inches='tight')
plt.show()
print(f'Saved → {out_path}')

## 8 — Game summary

In [ ]:
# Final board + score breakdown
board_abs = final_state.board.copy()
if final_player == -1: board_abs = -board_abs
black_stones, white_stones = count_territory(board_abs)
white_with_komi = white_stones + KOMI

fig, (ax_board, ax_bar) = plt.subplots(1, 2, figsize=(14, 7),
                                        gridspec_kw={'width_ratios': [1.2, 1]})

# Left: final board (larger for 19x19)
draw_board(ax_board, final_state, player=final_player,
           title=f'Final position  ({score_str})')
# Override font sizes for the summary board since it's bigger
ax_board.set_xticklabels(list(GTP_COLS), fontsize=7)
ax_board.set_yticklabels(range(1, BOARD_SIZE+1), fontsize=7)

# Right: score bar chart
categories = ['Black (area)', 'White (area + komi)']
values     = [black_stones, white_with_komi]
colors     = ['#1A1A1A', '#E8E8E0']
bars = ax_bar.barh(categories, values, color=colors,
                   edgecolor=['#888888', '#333333'], height=0.4)
for bar, val in zip(bars, values):
    ax_bar.text(val + 0.5, bar.get_y() + bar.get_height()/2,
                f'{val:.1f}', va='center', fontsize=12)
ax_bar.set_xlabel('Score (area + komi)', fontsize=11)
ax_bar.set_title('Score breakdown', fontsize=12)
ax_bar.set_xlim(0, max(values) * 1.2)
ax_bar.invert_yaxis()
ax_bar.grid(axis='x', alpha=0.3)

# Annotation: who won
winner_label = f'{winner_name} wins  ({score_str})'
az_result    = 'AlphaZero WIN' if az_won else 'AlphaZero LOSS'
ax_bar.text(0.5, -0.18, f'{winner_label}\n{az_result}',
            transform=ax_bar.transAxes, ha='center', fontsize=12,
            color='green' if az_won else 'red')

# Move-time stats
az_times = [f['elapsed_s'] for f in frames if f['is_az']]
kg_times = [f['elapsed_s'] for f in frames if not f['is_az']]
stats = (f'AlphaZero avg move: {np.mean(az_times):.1f}s  |  '
         f'KataGo avg move: {np.mean(kg_times):.1f}s') if az_times and kg_times else ''
fig.text(0.5, -0.04, stats, ha='center', fontsize=9, color='gray')

plt.tight_layout()
out_path = os.path.join(FIGURES_DIR, 'game_summary_19x19.png')
plt.savefig(out_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved → {out_path}')